In [ ]:
# Since some cells will fail on purpose in this notebook, only report minimal (meaningful) error messages
%xmode minimal

import os
# For a fair compiraison of performance against NumPy, we disable multithreading
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
os.environ["OMP_NUM_THREADS"] = "1"

import jax

# For a fair comparaison of performance against NumPy, we use doubles for JAX computations
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# `jax.vmap` - A powerfull JAX transformation
In this notebook, we will explore `jax.vmap`, a powerfull JAX transformation similar to Python's built-in `map` or NumPy's `np.vectorize`.

Let us have a small function which is applied on a scalar value and returns a scalar:

In [ ]:
# Scalar branching with a scalar -> works on one x, not on an array.
def f(x):
    idx = jnp.array(x >= 0).astype(int) + jnp.array(x >= 1).astype(int)# -> 0, 1, or 2
    return jax.lax.switch(idx, [lambda x: 0.,                # x < 0
                                lambda x: x**2,              # 0 <= x < 1
                                lambda x: 2.*x - 1.], x)     # x >= 1

This function uses `jax.lax.switch`, not covered in the previous BasicJAX notebook.
It is similar to `jax.lax.cond`, but with an arbitrary amount of branches ([link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.switch.html), read it!).

As `jax.lax.switch` chooses its branch given a scalar value (not broadcasted), this function will fail on arrays:

In [ ]:
print(f(1.5))
print(f(jnp.array([0., 1., 2., 3.])))

One solution is to use the `jax.vmap` transformation ([Link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html), read it!).

It takes a function and returns a new one that applies it across an axis of an array , semantically like mapping it over each element, but executed as a single (JIT compiled) vectorised pass rather than a Python loop.

`jax.vmap` works by batch tracing: it runs your function once on a batched tracer (an abstract value that stands in for the whole mapped axis rather than a single concrete element, similar to JIT tracer) and pushes that batch dimension through each primitive operation.
The result is vectorised array code produced in one pass, not a Python loop over the elements (like `np.vectorize` would do).
The practical consequence is that inside a vmapped function your argument is a tracer, not a value, so the same trace-time rules as `jit` apply: no Python control flow that branches on the mapped data.
Only the mapped axes are abstracted: arguments you leave unmapped (`in_axes=None`) stay concrete.

The new formed function can then be used to map arguments over:

In [ ]:
x = jnp.linspace(-0.5, 2., 1_000)
f_vmap = jax.vmap(f) # Vmap f
y = f_vmap(x)

As with any transformations, `vmap` can be composed with `jit`:

In [ ]:
f_vmap = jax.vmap(f)                  # No JIT
f_jit_vmap_in = jax.vmap(jax.jit(f))  # JIT on the function to vmap
f_jit_vmap_out = jax.jit(jax.vmap(f)) # JIT on the vmapped function

Timing:

In [ ]:
A = jnp.linspace(0.5, 2., 10_000)
%timeit f_vmap(A)
%timeit f_jit_vmap_in(A)
%timeit f_jit_vmap_out(A)

As can be seen, jitting the vmap function itself is one order of magnitude faster than only jitting its vmapped function, both orders of magnitude faster than eager mode JAX.

As other transformation `vmap` can be composed with is `jax.grad` which basically computes the gradient of a scalar quantity (explored more thoroughly in the dedicated autograd notebooks).
When vmapped over an array, it gives us the derivative:

In [ ]:
df_dx = jax.vmap(jax.grad(f))

dy_dx = df_dx(x)

plt.figure(figsize=(4., 3.), layout='constrained')
plt.plot(x, y, label="$f$")
plt.plot(x, dy_dx, label="$f'$")
plt.legend()
plt.show()

# Application - Pairwise distance

Suppose you have a set of $M$ $N$-dimensional points and want the distance between every pair: an $M \times M$ matrix $D$ where $D_{ij}$ is the Euclidean distance from point $x_i$ to point $x_j$.
The only thing really needed to build such matrix is the distance between two points:

$$D_{ij} = \sqrt{\sum_{k=1}^N(x_{ik}-x_{jk})^2}$$

Going from that to the full matrix can be painfull.
Writing a double `for` loop would work but would be very slow.
An other solution would be to reach for broadcasting with inserted axes: `X[:, None, :] - X[None, :, :]`, then reduce the right axis, which is correct but easy to get wrong.

`vmap` lets you skip all of that: keep the two-point function as-is and stack two maps
on top of it, one for each "loop".
The `in_axes` argument tells `vmap`, for each parameter, whether  sweep it (map over axis `0`) or hold it fixed (`None`):
- inner `vmap(dist, in_axes=(None, 0))` - hold row, sweep columns over all points -> one row
- outer `vmap(..., in_axes=(0, None))` - sweep rows over all points -> stack the rows

In [ ]:
# The only thing we write: distance between two single points.
def dist(x, y):
    return jnp.sqrt(jnp.sum((x - y)**2))

# Nested vmap turns it into the full matrix
dist_matrix_vmap = jax.jit(jax.vmap(jax.vmap(dist, in_axes=(None, 0)),  # inner: fix a, sweep b -> a row
                                                   in_axes=(0, None)))   # outer: sweep a        -> stack rows
points = jnp.array([[0., 0.],
                    [3., 0.],
                    [0., 4.],
                    [3., 4.]])

print(dist_matrix_vmap(points, points))

We now implement the broadcast version of the same code:

In [ ]:
@jax.jit
def dist_matrix_broadcast(points):
    return jnp.sqrt(jnp.sum((points[:, None, :] - points[None, :, :]) ** 2, axis=-1))

print(dist_matrix_broadcast(points))

We now compare performance of both versions on a big random array:

In [ ]:
N = 1_000
points = jax.random.normal(jax.random.key(1337), shape=(N, 3))

%timeit dist_matrix_vmap(points, points)
%timeit dist_matrix_broadcast(points)

As the benchmark shows, the broadcast version has a slight edge which is expected, since it maps directly onto primitive array operations while `vmap` layers a thin batching transformation on top.
The gap is small, though, and the vmapped version is the clearer and less error-prone of the two: you write only the two-point case and let the transformation handle the batching, with no axis bookkeeping to get wrong.

This barely scratches the surface: `vmap` is central to what makes JAX powerful
We'll return to it in laterexamples, some with no clean broadcasting counterpart at all (once we explore more JAX features).

# A small silly application - Seeing through noise

An other small application of `vmap`, this time somewhat more physical.

Given a set of readout samples in which each sample hides a sinusoidal signal buried in noise, we will try, by stacking these samples, to retrieve the frequency of the signal.

As for the previous example, we will compare performance of different implementations.

First define the sample parameters:

In [ ]:
sample_freq = 100        # Sampling frequency in Hz
n = 4096                 # Number of sample points
T = n/sample_freq        # Total time of the sample
dT = 1/sample_freq       # Sampling periode

f0 = 10                  # Frequency of our sinusoidal signal in Hz
A0 = 1.                  # Amplitude of our sinusoidal signal
noise_rms = 10.          # RMS noise

t = jnp.linspace(0., T, n)

## Generating samples

In this example, a sample is defined as a time serie of a sinusoidal signal of amplitude $A_0$ and frequency $f_0$ and burried in random Gaussian noise with a given standard deviation of $\sigma_n$.
In addition, each sample's signal will start at a random phase.

Each sample $i$ can then be written (in continuous time)
$$S_i(t)=A_0\sin\left(2\pi (f_0 t + \phi_i)\right) + \epsilon(t),$$
with
$$
\begin{align}
\epsilon & \sim \mathcal{N}(0, \sigma_n^2), \\
\Phi_i & \sim \mathcal{U}(0, 2\pi).
\end{align}
$$


In [ ]:
def generate_samples_numpy_naive(sample_count):
    samples = np.empty((sample_count, n))

    for i in range(sample_count):
        samples[i] = A0*np.sin(2.*jnp.pi*f0*t+np.random.uniform(high=2.*jnp.pi)) + noise_rms*np.random.normal(size=n)

    return samples

def generate_samples_numpy_broadcast(sample_count):
    return A0*np.sin(f0*t*2.*np.pi + np.random.uniform(high=2.*np.pi, size=(sample_count, 1))) + \
                    noise_rms*np.random.normal(size=(sample_count, n))

def generate_samples_naive(seed, sample_count):
    samples = jnp.empty((sample_count, n))    # Array of samples we will fill

    key_phase_noise, key_noise = jax.random.split(jax.random.key(seed), num=2)
    keys_phase_noise = jax.random.split(key_phase_noise, num=sample_count)
    keys_noise = jax.random.split(key_noise, num=sample_count)

    # Iteratively generate our samples.
    for i, (k_pn, k_n) in enumerate(zip(keys_phase_noise, keys_noise)):
        samples = samples.at[i].set(A0*jnp.sin(f0*t*2.*jnp.pi+jax.random.uniform(k_pn, maxval=2.*jnp.pi)) + noise_rms*jax.random.normal(k_n, shape=n))

    return samples

@jax.jit(static_argnames='sample_count')
def generate_samples_broadcast(seed, sample_count):
    key_phase_noise, key_noise = jax.random.split(jax.random.key(seed), num=2)
    return A0*jnp.sin(2.*jnp.pi*f0*t + jax.random.uniform(key_phase_noise, shape=(sample_count, 1), maxval=2.*jnp.pi)) + \
                    noise_rms*jax.random.normal(key_noise, shape=(sample_count, n))


@jax.jit(static_argnames='sample_count')
def generate_samples_vmap(seed, sample_count):
    key_phase_noise, key_noise = jax.random.split(jax.random.key(seed), num=2)
    keys_noise = jax.random.split(key_noise, num=sample_count)
    keys_phase_noise = jax.random.split(key_noise, num=sample_count)
    
    # Pure function which, for a given key, will sample a time serie
    def sample_serie(key_pn, key_n):
        return A0*jnp.sin(2.*jnp.pi*f0*t+jax.random.uniform(key_pn, maxval=2.*jnp.pi)) + noise_rms*jax.random.normal(key_n, shape=t.shape)

    # Generate all our sample in one go.
    return jax.vmap(sample_serie)(keys_phase_noise, keys_noise)


In [ ]:
sample_count = 1_000 # How many time series we simulate for the benchmark

%timeit generate_samples_numpy_naive(sample_count)
%timeit generate_samples_numpy_broadcast(sample_count)

%timeit generate_samples_naive(1337, sample_count)
%timeit generate_samples_broadcast(1337, sample_count).block_until_ready()
%timeit generate_samples_vmap(1337, sample_count).block_until_ready()


As in the pairwise example, the broadcasted version has a slight edge over the vmapped one.
Both are however faster than the NumPy ones.
Also, the JAX eager one should simply put in the trash.

> _**Question:**_ What happens when we try to jit the JAX eager version? Why?

We now look at one such time serie, and the mean of all samples.

In [ ]:
samples = generate_samples_broadcast(1337, 10_000)

# Inspect one time serie. Only noise seems visible
plt.subplots(ncols=2, nrows=1, figsize=(12., 3.), layout='constrained', sharey=False)
plt.subplot(1, 2, 1)
plt.plot(t[:128], samples[0, :128])
plt.xlabel("$t$ [s]")
plt.ylabel("Signal")
plt.title("Time serie")

plt.subplot(1, 2, 2)
plt.plot(t[:128], jnp.mean(samples, axis=0)[:128])
plt.xlabel("$t$ [s]")
plt.title("Mean time serie")
plt.show()

Because each series carries a random phase, averaging the time series directly makes the copies interfere destructively: the sinusoid averages toward zero and the waveform is genuinely lost (no amount of averaging in the time domain brings it back).
Its frequency and amplitude, however, survive.
The power spectrum discards phase which is each series own random $\Phi_i$ and is exactly what varies between series.
The magnitude hence the frequency and amplitude, is shared, so averaging the PSD is able to recover both.

We will now compute the mean PSD for a given set of time series.
Same as before, we compare several implementation (and forget about eager mode JAX):

In [ ]:
f = jnp.fft.rfftfreq(n, dT) # Frequency bins

# Using broadcasted NumPy
def compute_mean_psd_numpy_broadcast(samples):
    return np.mean(np.abs(np.fft.rfft(samples, axis=1))**2*dT/n, axis=0)

# Using JAX vmap
@jax.jit
def compute_mean_psd_vmap(samples):
    # Pure function which computes the spectrum of one given time serie
    def process_serie(sample_serie):
        return jnp.abs(jnp.fft.rfft(sample_serie))**2*dT/n
    
    # Vmap over all time series, then take the mean spectrum
    return jnp.mean(jax.vmap(process_serie)(samples) , axis=0)

@jax.jit
def compute_mean_psd_broadcast(samples):
    return jnp.mean(jnp.abs(jnp.fft.rfft(samples, axis=1))**2*dT/n, axis=0)

In [ ]:
%timeit compute_mean_psd_numpy_broadcast(samples)
%timeit compute_mean_psd_broadcast(samples).block_until_ready()
%timeit compute_mean_psd_vmap(samples).block_until_ready()

Here, the performance are somewhat similar between all versions, with a slight edge for JAX.

NumPy's FFT uses [PocketFFT](https://github.com/hayguen/pocketfft) as a backend, while JAX/XLA uses [DUCC](https://github.com/mreineck/ducc) which is itself based on PocketFFT and maintained by the same author (it originally used PocketFFT but has been switched to DUCC somewhat [recently](https://github.com/jax-ml/jax/pull/12122)).
As the compute time is here dominated by the FFT computations, timings are similar, with JAX's edge given by operation fusion.

> _**Question:**_ How does the timings evolve when we reduce the number of samples $n$? Why?

> _**Question:**_ How does the timings evolve when we enable multithreading? Why? (comment lines at the top code block)

For fun, we can look at the resulting spectrum and synthesize a new signal back in the time domain, revealing the sinusoid:

In [ ]:
psd = compute_mean_psd_broadcast(samples) # Estimate mean PSD
psd_signal = jnp.clip(psd - jnp.median(psd)-1., 0.) # Remove noise floor

# Invert signal PSD -> synthesize signal
signal = jnp.fft.irfft(jnp.sqrt(psd_signal*n/dT).astype(jnp.complex64), n=n)

plt.subplots(ncols=2, nrows=1, figsize=(10., 4.))
plt.subplot(1, 2, 1)
# We're actually plotting the signal autocorrelation.
# So we remove the first sample which is high due to residual (uncorrelated) noise.
plt.title("Signal time domain")
plt.plot(t[:128], signal[:128])

plt.subplot(1, 2, 2)
plt.title("Signal PSD")
plt.plot(f, psd_signal)
plt.show()